In [2]:
pip install lightgbm

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 11.1 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import lightgbm as lgb


In [2]:
#datacollection
dataset=pd.read_csv("insurance_pre.csv")
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [3]:
#preprocessing categorical data
dataset=pd.get_dummies(dataset,dtype=int,drop_first=True)
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [4]:
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [5]:
#spliting input and output data
input_data=dataset[['age', 'bmi', 'children','sex_male', 'smoker_yes']]
print(input_data)
output_data=dataset[['charges']]
print(output_data)

      age     bmi  children  sex_male  smoker_yes
0      19  27.900         0         0           1
1      18  33.770         1         1           0
2      28  33.000         3         1           0
3      33  22.705         0         1           0
4      32  28.880         0         1           0
...   ...     ...       ...       ...         ...
1333   50  30.970         3         1           0
1334   18  31.920         0         0           0
1335   18  36.850         0         0           0
1336   21  25.800         0         0           0
1337   61  29.070         0         0           1

[1338 rows x 5 columns]
          charges
0     16884.92400
1      1725.55230
2      4449.46200
3     21984.47061
4      3866.85520
...           ...
1333  10600.54830
1334   2205.98080
1335   1629.83350
1336   2007.94500
1337  29141.36030

[1338 rows x 1 columns]


In [6]:
#spliting training and test data
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(input_data,output_data,test_size=0.30,random_state=0)
print(x_train)
print(y_train)
print(x_test)
print(y_test)

      age     bmi  children  sex_male  smoker_yes
1163   18  28.215         0         0           0
196    39  32.800         0         0           0
438    52  46.750         5         0           0
183    44  26.410         0         0           0
1298   33  27.455         2         1           0
...   ...     ...       ...       ...         ...
763    27  26.030         0         1           0
835    42  35.970         2         1           0
1216   40  25.080         0         1           0
559    19  35.530         0         1           0
684    33  18.500         1         0           0

[936 rows x 5 columns]
          charges
1163   2200.83085
196    5649.71500
438   12592.53450
183    7419.47790
1298   5261.46945
...           ...
763    3070.80870
835    7160.33030
1216   5415.66120
559    1646.42970
684    4766.02200

[936 rows x 1 columns]
      age     bmi  children  sex_male  smoker_yes
578    52  30.200         1         1           0
610    47  29.370         1         

In [7]:
#standardizing the input values
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x_train=sc.fit_transform(x_train)
x_test=sc.transform(x_test)
print(x_train)
print(x_test)

[[-1.5330973  -0.40713453 -0.89833872 -0.97676557 -0.50466988]
 [-0.03364163  0.32855417 -0.89833872 -0.97676557 -0.50466988]
 [ 0.89459283  2.56690911  3.25603402 -0.97676557 -0.50466988]
 ...
 [ 0.03776102 -0.91016269 -0.89833872  1.02378711 -0.50466988]
 [-1.46169465  0.76659782 -0.89833872  1.02378711 -0.50466988]
 [-0.46205754 -1.96596021 -0.06746417 -0.97676557 -0.50466988]]
[[ 0.89459283 -0.08863026 -0.06746417  1.02378711 -0.50466988]
 [ 0.53757957 -0.22180837 -0.06746417 -0.97676557 -0.50466988]
 [ 0.60898222  1.57449152  0.76341038  1.02378711  1.98149332]
 ...
 [ 1.10880078  1.20785059 -0.89833872  1.02378711 -0.50466988]
 [ 1.75142463  1.34905148 -0.06746417  1.02378711 -0.50466988]
 [ 1.60861933 -0.92299913 -0.89833872 -0.97676557 -0.50466988]]


In [9]:
#LG boosting algorithm
from lightgbm import LGBMRegressor
regressor=LGBMRegressor(boosting_type='gbdt',num_leaves=31,max_depth=-1,learning_rate=0.1,n_estimators=100)
regressor=regressor.fit(x_train,y_train)

C:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000276 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 316
[LightGBM] [Info] Number of data points in the train set: 936, number of used features: 5
[LightGBM] [Info] Start training from score 13232.916456


In [10]:
y_predict=regressor.predict(x_test)
print(y_predict)
print(y_test)

[ 9991.6955793   8091.82232526 44242.48227634 12687.44522847
  9549.72199712  7018.83625386  2059.80986271 12836.38664824
  8154.60812735  5600.27468473  6995.75270097 16394.98768397
  8666.54783402  8467.84203633 21901.89116803 11282.34366058
 15600.731768    3049.48888456  5836.75691035 34326.88991511
 23764.69634507 17261.24792414 10063.61649756 26611.45880664
  4715.75225508  6176.54289302  5196.03595184  6820.32924539
  2412.06245212 11710.19835489  7319.78739296 49117.04832809
 17615.94190787 11732.22331163 17402.43556847  4461.7974098
 11660.99643497 37955.66338055 38472.29958032  3324.83909292
  6210.34268468  5844.37791358 19881.6746591  48842.85883941
 36669.5849967   3136.07901425 11784.5460969   8232.94767833
  5140.0193479  12138.94472456  1326.74223092  3068.28266267
 25245.61958372 46553.80261775 12667.31848606  8360.09980477
  4183.73558571 11380.41209627  8328.83045719 19089.35691893
  2113.8433097  45759.50229063 17170.82779452 16258.05591735
 12794.58199634  7953.629

In [11]:
from sklearn.metrics import r2_score
r_score=r2_score(y_test,y_predict)
print(r_score)

0.8699321391117371
